# 07 – Güven Skoru, Eksik Kavram Analizi ve Girdi Kontrolü

Puanlama modelimiz hazır. Ama bir öğretmenin sisteme güvenebilmesi için puan tek başına yetmez. Gereksinim analizinde (2. hafta) bunun için üç gereksinim tanımlamıştık:

| Gereksinim | Ne istiyor? | Bu not defterinde |
|---|---|---|
| **FR-04** Güven | Sistemin emin olamadığı cevaplar öğretmene işaretlenmeli | Bölüm A |
| **NFR-03** Açıklanabilirlik | Puanın dayandığı benzerlik ve **eksik kavramlar** gösterilmeli | Bölüm B |
| **FR-06** Girdi kontrolü | Boş, geçersiz veya hatalı biçimli cevaplar yakalanmalı | Bölüm C |

### Nasıl çalıştırılır?
1. **Önce 06 numaralı not defteri çalıştırılmış olmalı** (puanlar ve belirsizlikler oradan geliyor).
2. Dosyayı Drive'daki `CevapAnaliz` klasörüne yükle ve Colab'da aç. GPU önerilir (eksik kavram analizini hızlandırır): Çalışma zamanı → Çalışma zamanı türünü değiştir → T4 GPU.
3. **Çalışma zamanı → Tümünü çalıştır.** GPU ile yaklaşık 5 dakika sürer.

## 1. Kurulum

In [ ]:
import os, sys, subprocess
from pathlib import Path

COLAB = "google.colab" in sys.modules
if COLAB:
    from google.colab import drive
    drive.mount("/content/drive")
    DRIVE = Path("/content/drive/MyDrive/CevapAnaliz")
    PROJE = Path("/content/CevapAnaliz")
    if PROJE.exists():
        subprocess.run(["git", "-C", str(PROJE), "pull", "-q"], check=True)
    else:
        subprocess.run(["git", "clone", "-q",
                        "https://github.com/Semantik-Muhendisleri/CevapAnaliz.git", str(PROJE)], check=True)
    (DRIVE / "data").mkdir(parents=True, exist_ok=True)
    if not (PROJE / "data").exists():
        os.symlink(DRIVE / "data", PROJE / "data")
else:  # Kendi bilgisayarında çalıştırılırsa
    PROJE = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
    DRIVE = PROJE

os.chdir(PROJE)
sys.path.insert(0, str(PROJE))
SEKIL = DRIVE / "sekiller"; SEKIL.mkdir(parents=True, exist_ok=True)
SONUC = DRIVE / "sonuclar"; SONUC.mkdir(parents=True, exist_ok=True)

import numpy as np
import pandas as pd
import torch
HIZLI_TEST = os.environ.get("CEVAPANALIZ_HIZLI_TEST") == "1"   # yalnızca geliştirici testi için
CIHAZ = "cuda" if torch.cuda.is_available() else "cpu"

tahmin_dosyasi = SONUC / "06_tahminler.csv"
if not tahmin_dosyasi.exists():
    raise FileNotFoundError("06 numaralı not defterinin sonuçları bulunamadı. Önce 06'yı çalıştır.")
tahminler = pd.read_csv(tahmin_dosyasi, index_col=0, keep_default_na=False)
df = pd.read_csv(PROJE / "data" / "processed" / "temiz_veri.csv", index_col=0, keep_default_na=False)
tahminler["hata"] = (tahminler["tahmin"] - tahminler["puan"]).abs()
print(f"{len(tahminler):,} test cevabının puanı ve belirsizliği yüklendi.".replace(",", "."))

# Bölüm A – Güven skoru (FR-04)

06'da her test cevabı için bir **belirsizlik** değeri hesaplamıştık: Model aynı cevabı dropout açıkken 10 kez puanladığında puanlar ne kadar değişiyor? Değişim büyükse model emin değil demektir.

Fikir basit: **En belirsiz cevapları öğretmene gönderelim, gerisini sistem puanlasın.** Eğer belirsizlik gerçekten işe yarıyorsa, öğretmene gönderilen cevaplarda modelin hatası büyük, sistemde kalan cevaplarda küçük olmalıdır.

Bunu iki şekilde kontrol ediyoruz:
1. **Belirsizlik ile hata arasında ilişki var mı?** (Spearman korelasyonu; artı ve büyük olmalı)
2. **Cevapların %10'unu, %25'ini öğretmene gönderirsek sistemde kalan cevaplarda hata ne kadar azalıyor?** Karşılaştırma için aynı sayıda cevabı **rastgele** göndermeyi de deniyoruz. Belirsizliğe göre göndermek rastgeleden belirgin şekilde iyi değilse belirsizlik işe yaramıyor demektir.

> **Ortalama mutlak hata (MAE):** Sistemin puanı ile öğretmen puanı arasındaki farkın ortalaması (0–1 ölçeğinde). Küçük olması iyidir; 0,10 hata, 5 puanlık ölçekte ortalama 0,5 puanlık sapma demektir.

Hedefimiz (1. hafta, H5): Cevapların **en fazla %25'ini** öğretmene göndererek sistemde kalan cevaplarda hatayı belirgin şekilde azaltmak.

In [ ]:
from scipy.stats import spearmanr
from src.puanlama.olcutler import olc

GRUPLAR = {"Mohler": (tahminler["veri_seti"] == "mohler", 5),
           "SciEntsBank ve Beetle": (tahminler["veri_seti"] != "mohler", 2)}

print("Belirsizlik ile hata arasındaki Spearman korelasyonu:")
for ad, (secim, _) in GRUPLAR.items():
    t = tahminler[secim]
    print(f"  {ad}: {spearmanr(t['belirsizlik'], t['hata'])[0]:.2f}".replace(".", ","))

ORANLAR = np.round(np.arange(0, 0.51, 0.05), 2)
rastgele = np.random.default_rng(0)
egri, ozet = [], []
for ad, (secim, seviye) in GRUPLAR.items():
    t = tahminler[secim].sort_values("belirsizlik", ascending=False)   # en belirsizler başta
    for oran in ORANLAR:
        k = int(round(oran * len(t)))
        kalan, giden = t.iloc[k:], t.iloc[:k]
        rastgele_mae = np.mean([t["hata"].values[rastgele.permutation(len(t))[k:]].mean() for _ in range(50)])
        egri.append({"grup": ad, "oran": oran, "belirsizliğe göre": kalan["hata"].mean(), "rastgele": rastgele_mae})
        if oran in (0.0, 0.10, 0.25):
            o = olc(kalan["puan"], kalan["tahmin"], seviye)
            ozet.append({"grup": ad, "öğretmene giden": f"%{oran * 100:.0f}", "sistemde kalan cevap": len(kalan),
                         "kalanlarda MAE": o["mae"], "kalanlarda Pearson": o["pearson"], "kalanlarda QWK": o["qwk"],
                         "gidenlerde MAE": giden["hata"].mean() if k else np.nan})
egri, ozet = pd.DataFrame(egri), pd.DataFrame(ozet)
ozet.round(3).to_csv(SONUC / "07_guven_skoru.csv", index=False, encoding="utf-8")
display(ozet.round(3))

In [ ]:
import matplotlib.pyplot as plt

fig, eksenler = plt.subplots(1, 2, figsize=(14, 4.5))
for ax, ad in zip(eksenler, GRUPLAR):
    e = egri[egri["grup"] == ad]
    ax.plot(e["oran"] * 100, e["belirsizliğe göre"], color="#2a78d6", linewidth=2, marker="o", markersize=5,
            label="En belirsizleri öğretmene gönder")
    ax.plot(e["oran"] * 100, e["rastgele"], color="#b5b4ad", linewidth=2, linestyle="--",
            label="Rastgele gönder (karşılaştırma)")
    ax.axvline(25, color="#52514e", linewidth=1, linestyle=":")
    ax.text(25.5, ax.get_ylim()[1], "%25 sınırı", va="top", fontsize=9, color="#52514e")
    ax.set_title(ad, loc="left", fontsize=12)
    ax.set_xlabel("Öğretmene gönderilen cevapların oranı (%)")
    ax.set_ylabel("Sistemde kalan cevaplarda ortalama hata (MAE)")
    for kenar in ["top", "right"]:
        ax.spines[kenar].set_visible(False)
    ax.grid(color="#e6e5e0"); ax.set_axisbelow(True)
eksenler[0].legend(frameon=False, fontsize=9)
fig.suptitle("Güven skoru: Belirsiz cevapları öğretmene göndermek hatayı ne kadar azaltıyor?",
             x=0.01, ha="left", fontsize=13, fontweight="bold")
fig.tight_layout()
fig.savefig(SEKIL / "07_guven_skoru.png", dpi=150, bbox_inches="tight")
plt.show()

In [ ]:
def virgul(x, b=2):
    return f"{x:.{b}f}".replace(".", ",")

print("Yorum:")
for ad in GRUPLAR:
    o = ozet[ozet["grup"] == ad].set_index("öğretmene giden")
    e = egri[(egri["grup"] == ad) & (egri["oran"] == 0.25)].iloc[0]
    once, sonra = o.loc["%0", "kalanlarda MAE"], o.loc["%25", "kalanlarda MAE"]
    print(f"  {ad}: Cevapların %25'i öğretmene gönderilince sistemde kalan cevaplarda ortalama hata "
          f"{virgul(once, 3)} → {virgul(sonra, 3)} (%{(1 - sonra / once) * 100:.0f} azalma). "
          f"Rastgele gönderilseydi {virgul(e['rastgele'], 3)} olurdu. "
          f"Öğretmene gönderilen cevaplarda modelin hatası {virgul(o.loc['%25', 'gidenlerde MAE'], 3)}.")

**Grafiği nasıl okuyalım?** Mavi çizgi aşağı iniyorsa, belirsiz cevapları öğretmene gönderdikçe sistemde kalan cevaplardaki hata azalıyor demektir. Gri kesikli çizgi (rastgele gönderme) neredeyse düz kalır; çünkü rastgele seçilen cevaplar ortalama zorluktadır. **İki çizgi arasındaki fark, belirsizlik ölçüsünün ne kadar işe yaradığını gösterir.**

Sistemde bu şöyle kullanılacak: Belirsizliği belirli bir eşiğin üstünde olan cevaplar sonuç ekranında **"öğretmen incelemesi gerekli"** diye işaretlenecek (FR-04) ve öğretmen bu cevapları filtreleyebilecek (FR-08).

# Bölüm B – Eksik kavram analizi (NFR-03)

Öğretmene ve öğrenciye sadece "3/5" demek yetmez; **neyin eksik olduğunu** da göstermeliyiz. Bunu şöyle yapıyoruz:

1. **Referans cevabı kavramlara bölüyoruz.** Cümleyi noktalama işaretlerinden ve "and", "or", "because" gibi bağlaçlardan ayırıyoruz. Örnek:
   - Referans: *"The harder mineral will leave a scratch on the less hard mineral. If the black mineral is harder, the brown mineral will have a scratch."*
   - Kavramlar: ① *the harder mineral will leave a scratch on the less hard mineral* ② *if the black mineral is harder* ③ *the brown mineral will have a scratch*
2. **Her kavramın öğrenci cevabında anlam olarak geçip geçmediğine bakıyoruz.** Öğrenci cevabını kayan pencerelerle küçük parçalara bölüp her parçayı Sentence-BERT (MiniLM) ile kavramla karşılaştırıyoruz. En benzer parçanın benzerliği bir **eşiğin** üstündeyse kavram "var" sayılıyor.
3. **Eşiği veriyle seçiyoruz:** SciEntsBank ve Beetle'ın eğitim kümelerinden alınan cevaplarda farklı eşikleri deneyip "kapsanan kavram oranı" ile öğretmen puanı arasındaki ilişkiyi en yüksek yapan eşiği seçiyoruz. Sonra bu eşiği test cevaplarında kullanıyoruz.

**Doğru çalışıp çalışmadığını nasıl anlarız?** Bu veri setlerinde "hangi kavram eksik" diye bir etiket yok. Ama yöntem doğru çalışıyorsa, **öğretmenden yüksek puan alan cevaplarda kapsanan kavram oranı da yüksek** olmalı. Bunu Spearman korelasyonuyla ölçüyoruz ve örneklere bakıyoruz.

In [ ]:
import re
from sentence_transformers import SentenceTransformer
from sklearn.feature_extraction.text import ENGLISH_STOP_WORDS

KORUNACAK = {"not", "no", "nor", "never", "none", "without", "more", "less", "most", "before", "after",
             "above", "below", "same", "only", "all"}
GEREKSIZ = set(ENGLISH_STOP_WORDS) - KORUNACAK
_KELIME = re.compile(r"[A-Za-z0-9']+")
_BOL = re.compile(r"[.;:!?]|,|\band\b|\bor\b|\bbecause\b|\bso\b", flags=re.I)

def kavramlar(referans):
    """Referans cevabı kavramlara böler. En az iki anlamlı kelime içermeyen parçalar (örneğin
    "Terminals 1, 2 and 3" listesindeki "2") bir sonraki parçayla birleştirilir."""
    parcalar = [_KELIME.findall(p) for p in _BOL.split(referans.replace("...", " "))]
    parcalar = [p for p in parcalar if p]
    anlamli = lambda p: sum(k.isalpha() and k.lower() not in GEREKSIZ for k in p)
    sonuc, bekleyen = [], []
    for p in parcalar:
        bekleyen += p
        if anlamli(bekleyen) >= 2:
            sonuc.append(" ".join(bekleyen)); bekleyen = []
    if bekleyen:
        if sonuc:
            sonuc[-1] += " " + " ".join(bekleyen)
        else:
            sonuc.append(" ".join(bekleyen))
    return sonuc or [referans]

def pencereler(cevap, uzunluk):
    """Öğrenci cevabını, verilen kelime uzunluğunda kayan parçalara böler."""
    k = _KELIME.findall(cevap)
    if len(k) <= uzunluk:
        return [" ".join(k)]
    return [" ".join(k[i:i + uzunluk]) for i in range(len(k) - uzunluk + 1)]

def pencere_uzunlugu(kavram):
    return max(3, min(10, len(kavram.split()) + 2))

model = SentenceTransformer("all-MiniLM-L6-v2", device=CIHAZ)

def kavram_benzerlikleri(tablo):
    """Her satır için referanstaki her kavramın, öğrenci cevabındaki en benzer parçayla benzerliğini hesaplar."""
    kavram_listesi = {r: kavramlar(r) for r in tablo["referans_cevap"].unique()}
    tum_kavram = sorted({k for v in kavram_listesi.values() for k in v})
    kv = dict(zip(tum_kavram, model.encode(tum_kavram, normalize_embeddings=True, batch_size=256)))
    parcalar = sorted({p for r, c in zip(tablo["referans_cevap"], tablo["ogrenci_cevap"])
                       for k in kavram_listesi[r] for p in pencereler(c, pencere_uzunlugu(k))})
    pv = dict(zip(parcalar, model.encode(parcalar, normalize_embeddings=True, batch_size=256)))
    sonuc = []
    for r, c in zip(tablo["referans_cevap"], tablo["ogrenci_cevap"]):
        sonuc.append([(k, float(np.max(np.stack([pv[p] for p in pencereler(c, pencere_uzunlugu(k))]) @ kv[k])))
                      for k in kavram_listesi[r]])
    return sonuc

for ornek in ["The harder mineral will leave a scratch on the less hard mineral. If the black mineral is harder, the brown mineral will have a scratch.",
              "Terminals 1, 2 and 3 are connected to the negative battery terminal",
              "The block inside a do...while statement will execute at least once."]:
    print(kavramlar(ornek))

In [ ]:
# 1) Eşik seçimi: SciEntsBank ve Beetle eğitim kümelerinden örnekler
n = 150 if HIZLI_TEST else 1500
secim = pd.concat([df[(df["veri_seti"] == v) & (df["bolum"] == "train")].sample(n, random_state=0)
                   for v in ["scientsbank", "beetle"]])
secim_benzerlik = kavram_benzerlikleri(secim)
ESIKLER = [0.40, 0.45, 0.50, 0.55, 0.60, 0.65, 0.70]
esik_tablosu = pd.Series({e: spearmanr([np.mean([b >= e for _, b in s]) for s in secim_benzerlik],
                                      secim["puan"])[0] for e in ESIKLER}, name="Spearman")
ESIK = float(esik_tablosu.idxmax())
print("Eşiklere göre kapsanan kavram oranı ile öğretmen puanı arasındaki Spearman (eğitim örnekleri):")
display(esik_tablosu.round(3).to_frame())
print(f"Seçilen eşik: {ESIK:.2f}".replace(".", ","))

In [ ]:
# 2) Test cevaplarında uygulama
test = df.loc[tahminler.index]
if HIZLI_TEST:
    test = test.groupby("veri_seti", group_keys=False).head(150)
test_benzerlik = kavram_benzerlikleri(test)
test = test.assign(kavramlar=[[k for k, _ in s] for s in test_benzerlik],
                   eksik=[[k for k, b in s if b < ESIK] for s in test_benzerlik],
                   kapsama=[np.mean([b >= ESIK for _, b in s]) for s in test_benzerlik])

satirlar = []
for v, ad in [("mohler", "Mohler"), ("scientsbank", "SciEntsBank"), ("beetle", "Beetle")]:
    t = test[test["veri_seti"] == v]
    satirlar.append({"veri seti": ad, "cevap": len(t), "referans başına kavram": t["kavramlar"].map(len).mean(),
                     "ortalama kapsama": t["kapsama"].mean(),
                     "kapsama–öğretmen puanı Spearman": spearmanr(t["kapsama"], t["puan"])[0]})
kavram_ozet = pd.DataFrame(satirlar)
kavram_ozet.round(3).to_csv(SONUC / "07_eksik_kavram.csv", index=False, encoding="utf-8")
display(kavram_ozet.round(2))

def puan_grubu(satir):
    if satir["veri_seti"] == "mohler":
        p = satir["puan"] * 5
        return "Mohler 0–2" if p <= 2 else "Mohler 2,5–4" if p <= 4 else "Mohler 4,5–5"
    return {0.0: "SemEval yanlış", 0.5: "SemEval kısmen doğru", 1.0: "SemEval doğru"}[satir["puan"]]
test["grup"] = test.apply(puan_grubu, axis=1)
SIRA = ["Mohler 0–2", "Mohler 2,5–4", "Mohler 4,5–5", "SemEval yanlış", "SemEval kısmen doğru", "SemEval doğru"]
grup_kapsama = test.groupby("grup")["kapsama"].mean().reindex(SIRA)

import matplotlib.pyplot as plt
fig, ax = plt.subplots(figsize=(11, 4))
renkler = ["#2a78d6"] * 3 + ["#eb6834"] * 3
cubuk = ax.bar(range(len(SIRA)), grup_kapsama.values * 100, color=renkler, width=0.6)
for x, d in enumerate(grup_kapsama.values):
    ax.text(x, d * 100 + 1.5, f"%{d * 100:.0f}", ha="center", fontsize=10)
ax.set_xticks(range(len(SIRA)), [s.replace(" ", "\n", 1) for s in SIRA])
ax.set_ylabel("Kapsanan kavram oranı (%)")
ax.set_ylim(0, 105)
for kenar in ["top", "right"]:
    ax.spines[kenar].set_visible(False)
ax.grid(axis="y", color="#e6e5e0"); ax.set_axisbelow(True)
ax.set_title("Öğretmen puanına göre ortalama kapsanan kavram oranı (test cevapları)", loc="left",
             fontsize=12, fontweight="bold")
fig.tight_layout()
fig.savefig(SEKIL / "07_eksik_kavram.png", dpi=150, bbox_inches="tight")
plt.show()

**Ne bekliyoruz?** Öğretmen puanı arttıkça kapsanan kavram oranı da artmalı (grafikte çubuklar soldan sağa yükselmeli) ve Spearman değerleri artı olmalı. Bu, yöntemin eksik kavramları anlamlı şekilde bulduğunu gösterir.

Şimdi sistemin öğretmene ve öğrenciye göstereceği açıklamaya örneklerle bakalım. ✓ = cevapta var, ✗ = eksik.

In [ ]:
import textwrap

def acikla(satir):
    print("=" * 110)
    print(textwrap.fill("Referans: " + satir["referans_cevap"], 110, subsequent_indent="          "))
    print(textwrap.fill("Öğrenci : " + satir["ogrenci_cevap"], 110, subsequent_indent="          "))
    print(f"Öğretmen puanı: {satir['puan']:.2f}   Sistemin puanı: {tahminler.loc[satir.name, 'tahmin']:.2f}   "
          f"Belirsizlik: {tahminler.loc[satir.name, 'belirsizlik']:.3f}".replace(".", ","))
    for k in satir["kavramlar"]:
        print(f"   {'✗ EKSİK' if k in satir['eksik'] else '✓ var  '}  {k}")

ornek = test[test["kavramlar"].map(len) >= 2]
for v in ["mohler", "scientsbank", "beetle"]:
    t = ornek[ornek["veri_seti"] == v]
    for secilen in [t[t["kapsama"] < 1].sample(1, random_state=1), t[t["kapsama"] == 1].sample(1, random_state=1)]:
        if len(secilen):
            acikla(secilen.iloc[0])

**Bilinen bir zayıflık:** Kavram analizi Sentence-BERT kullandığı için, 03'te gördüğümüz sorunu burada da taşır: Aynı konudan bahseden ama **tersini söyleyen** bir parçayı kavramın karşılığı sanabilir. Aşağıda öğretmenin **çelişkili** dediği ama bütün kavramları "var" görünen bir örnek var:

In [ ]:
zaaf = test[(test["orijinal_puan"] == "contradictory") & (test["kapsama"] == 1)]
print(f"Çelişkili etiketli test cevaplarının %{len(zaaf) / max(1, (test['orijinal_puan'] == 'contradictory').sum()) * 100:.0f}'"
      "inde bütün kavramlar 'var' görünüyor.")
if len(zaaf):
    acikla(zaaf.sample(1, random_state=0).iloc[0])

Bu yüzden açıklama bölümü **tek başına puan vermek için değil**, ince ayarlı modelin verdiği puanın yanında öğretmene ve öğrenciye **yol gösteren bir ipucu** olarak kullanılacak. Puanı ince ayarlı model veriyor; o model zıtlıkları büyük ölçüde ayırt edebiliyor (05 ve 06).

# Bölüm C – CSV girdi kontrolü (FR-06)

Öğretmen öğrenci cevaplarını CSV dosyasıyla yükleyecek. Dosyada hata olabilir: eksik sütun, boş cevap, sistemde olmayan bir soru numarası, aynı öğrencinin aynı soruya iki cevabı… 3. hafta raporunda tanımladığımız kurallar `src/girdi/csv_kontrol.py` dosyasında kodlandı ve birim testleriyle doğrulandı. Kurallar:
- Dosya CSV olmalı, boş olmamalı, 5 MB'tan küçük olmalı; Türkçe Excel kodlaması (cp1254) ve noktalı virgül ayırıcı da kabul edilir.
- `ogrenci_no`, `soru_no`, `cevap` sütunları zorunlu; eksikse dosyanın tamamı reddedilir.
- Boş cevaplar, kayıtlı olmayan sorular ve tekrarlanan satırlar **puanlanmaz** ve öğretmene satır numarasıyla gösterilir; geri kalan satırlar puanlanmaya devam eder (NFR-06).

Hatalı bir örnek dosyayla deneyelim:

In [ ]:
from src.girdi.csv_kontrol import csv_kontrol

ornek_dosya = """ogrenci_no,soru_no,cevap
101,S1,A prototype simulates the behaviour of part of the software.
102,S1,
103,S7,Abstraction and reusability.
104,S2,Data members and member functions.
104,S2,Data members and functions.
,S2,Class variables.
"""
sonuc = csv_kontrol(ornek_dosya.encode("utf-8"), "ogrenci_cevaplari.csv", kayitli_sorular={"S1", "S2"})
print(f"Puanlanacak satır sayısı: {len(sonuc.gecerli)}")
display(sonuc.gecerli)
print("Öğretmene gösterilecek uyarılar:")
display(pd.DataFrame(sonuc.hatalar))

print("\nEksik sütunlu bir dosya:")
print(" ", csv_kontrol(b"ogrenci_no,cevap\n101,Cevap\n", "eksik.csv", {"S1"}).dosya_hatasi)
print("Excel dosyası:")
print(" ", csv_kontrol(b"...", "cevaplar.xlsx", {"S1"}).dosya_hatasi)

# Özet

| Gereksinim | Ne yapıldı? | Sonuç dosyası |
|---|---|---|
| **FR-04** Güven | Monte Carlo dropout ile her cevabın belirsizliği hesaplandı; en belirsiz %10 / %25 öğretmene gönderildiğinde sistemde kalan cevaplardaki hata ölçüldü | `sonuclar/07_guven_skoru.csv`, `sekiller/07_guven_skoru.png` |
| **NFR-03** Açıklanabilirlik | Referans cevap kavramlara bölündü; her kavramın öğrenci cevabında olup olmadığı Sentence-BERT ile bulundu; eşik eğitim verisinden seçildi | `sonuclar/07_eksik_kavram.csv`, `sekiller/07_eksik_kavram.png` |
| **FR-06** Girdi kontrolü | CSV kontrol kuralları kodlandı ve birim testleriyle doğrulandı; hatalı satırlar satır numarasıyla raporlanıyor | `src/girdi/csv_kontrol.py`, `tests/test_csv_kontrol.py` |

Bu üç parça, 3. haftada tasarlanan sistemde **güven kontrolü**, **açıklama** ve **cevap yükleme** bölümlerinin çekirdeğidir. Bir sonraki adım, bunları puanlama servisine ve arayüze bağlamaktır.

Sonuçları 6. hafta raporu için Drive'daki `sonuclar` ve `sekiller` klasörlerinden alabilirsin.